# SAM 3.1 tracking on Colab (free T4 GPU)

Role A / Phase 1. Run this with **Runtime > Change runtime type > T4 GPU** selected.

GPU reality check (see `docs/COMPUTE.md`): the hackathon provides no GPUs, RunPod needs a booth
visit, our only physical GPU (a K1900, ~2GB) is below SAM 3.1's ~4GB floor. Colab's free T4
(16GB, real CUDA) clears that easily and costs nothing — that's why we're here.

We use the **official 🤗 Transformers path** (`Sam3VideoModel`/`Sam3VideoProcessor`, straight
from the model card at https://huggingface.co/facebook/sam3) — weights auto-download via
`from_pretrained()` once your HF account has approved access, no manual file placement needed.

**Known risk:** SAM 3 weights are gated and access is **not guaranteed** — people have been
denied. If Step 3 below doesn't clear in time, stop and use the fal.ai path instead (no GPU
needed at all): see `docs/DEFERRED.md`. Or keep developing everything else against
`sam.backend: replay` (already verified, works today, no GPU).

In [ ]:
# 1. Confirm we actually got a GPU runtime (fails loud, exactly like src.utils.gpu)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Clone the repo and install deps (CUDA torch ships by default on Colab)
import os

REPO_URL = "https://github.com/wheredawoodat949/AI-Hackathon"
BRANCH = "feat/tracking-ashmeet"  # switch if you're testing a different branch

if not os.path.isdir("AI-Hackathon"):
    !git clone --branch {BRANCH} {REPO_URL}.git
%cd AI-Hackathon
!pip install -q -r requirements.txt
!pip install -q -U transformers accelerate  # SAM3 support needs a very recent transformers
!python -m src.utils.gpu        # must print CUDA — if not, fix the runtime type above

## 3. SAM 3 access (gated — the one step that can block this whole path)

1. Request access: https://huggingface.co/facebook/sam3 (sign the agreement). **Can be denied
   or take time to approve** — check status any time at https://huggingface.co/settings/gated-repos.
2. Create a read token: https://huggingface.co/settings/tokens
3. Paste it below and log in — `from_pretrained("facebook/sam3")` will then auto-download the
   weights the first time `sam_local.py` needs them. No manual file download/placement needed.

In [ ]:
import getpass

from huggingface_hub import login

login(token=getpass.getpass("HF token (input hidden): "))

In [ ]:
# Smoke test: does this account actually have approved access yet?
try:
    from transformers import Sam3VideoProcessor
    Sam3VideoProcessor.from_pretrained("facebook/sam3")
    print("Access OK — weights/processor load fine.")
except Exception as e:
    print("Not accessible yet:", e)
    print("Check https://huggingface.co/settings/gated-repos for approval status.")
    print("If this doesn't clear soon, see docs/DEFERRED.md (fal.ai) instead of waiting.")

In [ ]:
# 4. Get a real clip to track on (annotations-only download is fast; this adds the video).
!python -m src.data.download --match 117093

In [ ]:
# 5. Run real SAM 3.1 tracking and render an annotated clip — the actual Phase 1 deliverable.
import glob

video = sorted(glob.glob("data/videos/117093*"))[0]
print("tracking:", video)
!python -m src.tracking.demo --backend local --video {video} --seconds 10

In [ ]:
# 6. Sanity-check + preview the output (download it from the Colab file browser too).
import glob

out = sorted(glob.glob("outputs/track_117093_local*"))
print(out)

from IPython.display import Video
Video(out[0], embed=True, width=640) if out else None

## If this worked
Set `sam.backend: local` in `config.yaml` on the repo (or just keep using `--backend local`
from here), commit `outputs/track_117093_local.mp4` somewhere visible (it's gitignored —
download it locally / drop it in the team Drive), and tell Role B (pitch+eval) real tracked
detections are available for the homography + HOTA work.

## If Step 3 (access) never cleared
Don't burn more time waiting. Two real fallbacks, both already implemented:
- `docs/DEFERRED.md` → fal.ai hosted SAM 3.1, no GPU/gating, ~$0.16/clip.
- `sam.backend: replay` → already verified, runs the whole pipeline on GSR ground truth, no GPU.